Organizando Colunas

In [6]:
import pandas as pd
import os
import warnings

# Ignorar avisos do pandas sobre arquivos Excel
warnings.simplefilter(action='ignore', category=UserWarning)

caminho_pasta = r"C:\Users\user\Downloads\Script_Exames_HV"
arquivo_entrada = os.path.join(caminho_pasta, "EXAME.xlsx") 
arquivo_saida = os.path.join(caminho_pasta, "EXAME_Base_BI.xlsx")

def preparar_base_bi():
    print(f"Lendo o arquivo: {arquivo_entrada}...")
    
    try:
        df_raw = pd.read_excel(arquivo_entrada, sheet_name="GERAL", header=None)
        
        # 1. Capturar os ANOS na linha 1 (índice 0)
        anos = df_raw.iloc[0].ffill()
            
        # 2. Capturar os MESES na linha 2 (índice 1) com ffill
        meses = df_raw.iloc[1].ffill()
        
        # 3. Capturar as MÉTRICAS na linha 3 (índice 2)
        metricas_originais = df_raw.iloc[2]
        
        # 4. Limpar e padronizar as métricas
        metricas_limpas = []
        ultima_metrica = ""
        
        for m in metricas_originais:
            m_str = str(m).strip().upper()
            if m_str in ['CONS', 'CONSULTAS']:
                m_limpa = 'Consultas'
                ultima_metrica = m_limpa
            elif m_str in ['EX', 'EXAMES']:
                m_limpa = 'Exames'
                ultima_metrica = m_limpa
            elif m_str in ['CIR', 'CIRURGIAS']:
                m_limpa = 'Cirurgias'
                ultima_metrica = m_limpa
            elif m_str == '%':
                m_limpa = f'% de {ultima_metrica}'
            else:
                m_limpa = str(m).strip()
                ultima_metrica = m_limpa
                
            metricas_limpas.append(m_limpa)
            
        # 5. Extrair os dados (Unpivot)
        dados_estruturados = []
        
        # Os médicos começam na linha 4 (índice 3)
        for index in range(3, len(df_raw)):
            linha = df_raw.iloc[index]
            medico = linha[0]
            
            # Pular linhas vazias
            if pd.isna(medico) or str(medico).strip() == "":
                continue
                
            for col in range(1, len(linha)):
                ano_raw = anos[col]
                
                # AJUSTE AQUI: Tenta converter para float e depois int para garantir o formato de número
                try:
                    ano = int(float(ano_raw))
                except (ValueError, TypeError):
                    ano = ano_raw # Mantém original caso seja algum texto de cabeçalho
                    
                mes = str(meses[col]).strip()
                metrica = metricas_limpas[col]
                valor = linha[col]
                
                # Só adiciona se houver mês e métrica válidos
                if mes.upper() != "NAN" and metrica.upper() != "NAN" and pd.notna(valor):
                    dados_estruturados.append({
                        "Ano": ano, # Agora entra como número inteiro
                        "Mês": mes,
                        "Médico": str(medico).strip(),
                        "Métrica": metrica,
                        "Valor": valor
                    })
                    
        # 6. Pivotar para o formato final
        df_long = pd.DataFrame(dados_estruturados)
        
        df_bi = df_long.pivot_table(
            index=['Ano', 'Mês', 'Médico'], 
            columns='Métrica', 
            values='Valor', 
            aggfunc='first'
        ).reset_index()
        
        # Preencher vazios com ZERO
        df_bi = df_bi.fillna(0)
        
        # 7. Ordenar as colunas logicamente
        colunas_ordem = ['Ano', 'Mês', 'Médico', 'Consultas', '% de Consultas', 'Exames', '% de Exames', 'Cirurgias', '% de Cirurgias', 'PROC']
        colunas_existentes = [c for c in colunas_ordem if c in df_bi.columns]
        colunas_extras = [c for c in df_bi.columns if c not in colunas_existentes]
        
        df_bi = df_bi[colunas_existentes + colunas_extras]
        
        # 8. Salvar
        df_bi.to_excel(arquivo_saida, index=False)
        print(f"Sucesso! Base padronizada gerada em: {arquivo_saida}")
        
    except Exception as e:
        print(f"Ocorreu um erro: {e}")

if __name__ == "__main__":
    preparar_base_bi()

Lendo o arquivo: C:\Users\user\Downloads\Script_Exames_HV\EXAME.xlsx...
Sucesso! Base padronizada gerada em: C:\Users\user\Downloads\Script_Exames_HV\EXAME_Base_BI.xlsx


C:\Users\user\AppData\Local\Temp\ipykernel_26356\1560852572.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  anos = df_raw.iloc[0].ffill()
